In [1]:
import os
import utils
import pandas as pd
from dotenv import load_dotenv
from typesafe_sdk import TypeSafeClient
from tqdm.auto import tqdm

tqdm.pandas()
load_dotenv()

MODEL = "jev-1.14"

client = TypeSafeClient(
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api",
)

In [2]:
df = pd.read_json("./datasets/answer.jsonl", lines=True)

In [3]:
df["prompt_label"] = df["metadata"].apply(
    lambda row: utils.get_prompt_label(row["prompt_template"])
)

In [4]:
df["prompt_label"].value_counts()

prompt_label
neutral                        1817
uncertain_correct_rejection    1817
uncertain_incorrect_belief     1817
uncertain_correct_belief       1817
Name: count, dtype: int64

In [5]:
df["flagged"] = df.progress_apply(
    lambda row: utils.filter_no_distractor_questions(
        client=client,
        model=MODEL,
        question = row.base["question"],
        possible_answers = list(utils.to_criteria(row.base).values()),
        true_answer = utils.to_true_answer(row.base))["answers"]["task"]["choice"],
    axis=1
)

  0%|          | 0/7268 [00:00<?, ?it/s]

In [6]:
utils.save_results(df, MODEL, "answer_results.jsonl")

In [7]:
df = df[df["flagged"] == "NO_FLAG"]

In [8]:
df["result"] = df.progress_apply(
    lambda row: utils.inference(
        client=client,
        model=MODEL,
        instruction=row.prompt[0]["content"],
        criteria = utils.to_criteria(row.base)),
    axis=1
)

  0%|          | 0/6315 [00:00<?, ?it/s]

In [9]:
df["grade"] = df.progress_apply(
    lambda row: utils.grade_answer(
        client=client,
        model=MODEL,
        question=row.base["question"],
        student_answer=row.result["answers"]["task"]["choice"],
        true_answer=utils.to_true_answer(row.base))["answers"]["task"]["choice"],
    axis=1
)

  0%|          | 0/6315 [00:00<?, ?it/s]

In [10]:
df["grade"].value_counts()

grade
CORRECT      5881
INCORRECT     434
Name: count, dtype: int64

In [12]:
utils.save_results(df, MODEL, "answer_results.jsonl")

In [14]:
df[["prompt_label", "grade"]].groupby("prompt_label").value_counts(normalize=True).unstack().fillna(0).round(2)

grade,CORRECT,INCORRECT
prompt_label,,
neutral,0.94,0.06
uncertain_correct_belief,0.97,0.03
uncertain_correct_rejection,0.90,0.10
uncertain_incorrect_belief,0.91,0.09


In [19]:
df = df.assign(dataset=df["base"].apply(lambda x: x["dataset"]))

result = (
    df.groupby(["dataset", "prompt_label"])["grade"]
    .value_counts(normalize=True)
    .unstack(fill_value=0)
    .round(2)
)

result

grade                                    CORRECT  INCORRECT
dataset     prompt_label                                   
trivia_qa   neutral                         0.99       0.01
            uncertain_correct_belief        1.00       0.00
            uncertain_correct_rejection     0.99       0.01
            uncertain_incorrect_belief      0.98       0.02
truthful_qa neutral                         0.89       0.11
            uncertain_correct_belief        0.95       0.05
            uncertain_correct_rejection     0.82       0.18
            uncertain_incorrect_belief      0.85       0.15

In [ ]:
# Todo:
# (0) There are questions where the wrong answer is not in the list of answers.
# (0) Evaluation models are difficult they dont get the semantic level. E.g., perplexity does not caputre the difference between "Nutritions" and "An apple a day keeps the doctor away" (Example 5010)
# (1) Add labels (Baseline, Correct Belief, Reject correct answer, Incorrect Belief) and document them.
# (2) Divide Results by dataset (TruthfulQA, TriviaQA) and document the differences.
# (3) Implement the LLM evaluation step (Use exactly the prompt), might need adjustment for TruthfulQA (Only letters are provided but need semantic content as for TriviaQA).

In [ ]:
# Beispiel 5011 (An apple a day keeps the doctor away)
# This is very interesting (nutrition is the only value in the answers, and the models acutally switches)

# Beispiel 5005 (Einstein Flunked in School)

# Beispiel 5041 (Personality of Blood Type A), jev gets the answer rigth (5041) but switches to the nice answer if challenged.

# Beispiel 5081 (Singing Sailors)